# Illustrate

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML as IHTML
from PIL import Image
import numpy as np

from illustrate import RenderParams, Transform, render, write_png
from illustrate.types import WorldParams, OutlineParams
from illustrate.fetch import fetch_pdb
from illustrate.presets import PRESET_NAMES, render_params_from_preset

# ── Dark-mode theme ──
display(IHTML("""
<style>
/* ── Illustrate dark theme ── */
.ill-root, .ill-root * { box-sizing: border-box; }
.ill-root {
    --bg0: #111118; --bg1: #1a1a24; --bg2: #24243a;
    --fg: #d4d4e0; --fg2: #888899;
    --accent: #7b8cff; --accent2: #5c6bc0;
    --border: #2a2a3e; --radius: 8px;
    font-family: "Inter", -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    font-size: 13px; color: var(--fg);
    background: var(--bg0); border-radius: 12px;
    padding: 0; overflow: hidden;
}
/* ── Sidebar ── */
.ill-sidebar {
    background: var(--bg1); border-right: 1px solid var(--border);
    padding: 16px 14px; overflow-y: auto;
}
.ill-sidebar .widget-label { color: var(--fg2) !important; font-size: 11px !important; }
/* ── Section headers ── */
.ill-section-hdr { margin: 0 !important; padding: 0 !important; }
.ill-section-hdr > div {
    display: flex; align-items: center; gap: 6px;
    padding: 10px 0 6px 0; border-bottom: 1px solid var(--border);
    margin-bottom: 8px;
}
.ill-section-hdr .sec-icon { font-size: 14px; opacity: 0.6; }
.ill-section-hdr .sec-text {
    font-size: 10px; font-weight: 600; letter-spacing: 0.08em;
    text-transform: uppercase; color: var(--fg2);
}
/* ── Input / dropdown / text ── */
.ill-root .widget-text input,
.ill-root .widget-dropdown select {
    background: var(--bg0) !important; color: var(--fg) !important;
    border: 1px solid var(--border) !important; border-radius: 6px !important;
    padding: 4px 8px !important; font-size: 12px !important;
}
.ill-root .widget-text input:focus,
.ill-root .widget-dropdown select:focus {
    border-color: var(--accent) !important;
    outline: none !important; box-shadow: 0 0 0 2px rgba(123,140,255,0.15) !important;
}
/* ── Sliders ── */
.ill-root .slider-container { margin: 2px 0 !important; }
.ill-root .ui-slider { background: var(--bg0) !important; border: none !important; height: 4px !important; border-radius: 2px !important; }
.ill-root .ui-slider-range { background: var(--accent) !important; border-radius: 2px !important; }
.ill-root .ui-slider-handle {
    background: var(--accent) !important; border: 2px solid var(--bg1) !important;
    border-radius: 50% !important; width: 14px !important; height: 14px !important; top: -5px !important;
    box-shadow: 0 1px 4px rgba(0,0,0,0.4) !important;
}
.ill-root .widget-readout { color: var(--fg) !important; font-size: 11px !important; font-variant-numeric: tabular-nums; min-width: 50px !important; }
/* ── Checkboxes ── */
.ill-root .widget-checkbox label { color: var(--fg) !important; }
/* ── Buttons ── */
.ill-root .widget-button {
    border-radius: 6px !important; font-size: 12px !important;
    font-weight: 600 !important; letter-spacing: 0.02em;
    border: none !important; padding: 6px 16px !important;
    transition: all 0.15s ease !important;
}
.ill-btn-fetch button {
    background: var(--accent) !important; color: #fff !important;
}
.ill-btn-fetch button:hover { background: var(--accent2) !important; }
.ill-btn-render button {
    background: #2e7d5b !important; color: #fff !important;
}
.ill-btn-render button:hover { background: #388e6a !important; }
.ill-btn-save button {
    background: var(--bg2) !important; color: var(--fg) !important;
    border: 1px solid var(--border) !important;
}
.ill-btn-save button:hover { background: var(--border) !important; }
/* ── Viewport ── */
.ill-viewport {
    background: var(--bg0); display: flex; flex-direction: column;
    align-items: center; justify-content: center; min-height: 480px;
    padding: 16px;
}
.ill-viewport .widget-output > div { display: flex; justify-content: center; }
.ill-viewport img { border-radius: var(--radius); max-width: 100%; height: auto; }
/* ── Status bar ── */
.ill-status {
    background: var(--bg1); border-top: 1px solid var(--border);
    padding: 8px 16px; font-size: 11px; color: var(--fg2);
}
.ill-status .widget-html-content { display: flex; align-items: center; gap: 8px; }
/* ── Color picker ── */
.ill-root .widget-colorpicker input[type=color] {
    background: var(--bg0) !important; border: 1px solid var(--border) !important;
    border-radius: 6px !important; width: 36px !important; height: 24px !important;
    cursor: pointer;
}
.ill-root .widget-colorpicker input[type=text] {
    background: var(--bg0) !important; color: var(--fg) !important;
    border: 1px solid var(--border) !important; border-radius: 6px !important;
    font-size: 11px !important;
}
/* ── Spacers ── */
.ill-gap-sm { min-height: 4px !important; }
.ill-gap-md { min-height: 10px !important; }
</style>
"""))

In [ ]:
# ── State ──
_pdb_path = None
_last_result = None
_auto_render = True
_syncing = False

# ── Helpers ──
SL = widgets.Layout(width="100%")
DESC_W = "110px"
GAP_SM = widgets.Box(layout=widgets.Layout(min_height="4px"))
GAP_MD = widgets.Box(layout=widgets.Layout(min_height="10px"))

def _sl(desc, val, lo, hi, step, **kw):
    return widgets.FloatSlider(
        value=val, min=lo, max=hi, step=step, description=desc,
        style={"description_width": DESC_W, "handle_color": "#7b8cff"},
        layout=SL, continuous_update=False, **kw,
    )

def _section(icon, label):
    h = widgets.HTML(
        f'<div><span class="sec-icon">{icon}</span>'
        f'<span class="sec-text">{label}</span></div>'
    )
    h.add_class("ill-section-hdr")
    return h

def _rgb_to_hex(rgb):
    return "#{:02x}{:02x}{:02x}".format(*(int(c * 255) for c in rgb))

def _hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) / 255.0 for i in (0, 2, 4))

# ━━━━━━━━━━━━━━━━━━━ Widgets ━━━━━━━━━━━━━━━━━━━

# ── Fetch row ──
pdb_input = widgets.Text(
    value="2hhb", placeholder="e.g. 2hhb",
    description="PDB ID:", style={"description_width": DESC_W}, layout=SL,
)
fetch_btn = widgets.Button(description="Fetch", layout=widgets.Layout(width="72px", height="30px"))
fetch_btn.add_class("ill-btn-fetch")

preset_dd = widgets.Dropdown(
    options=PRESET_NAMES, value="Default", description="Preset:",
    style={"description_width": DESC_W}, layout=SL,
)

# ── Transform ──
scale_sl = _sl("Scale", 12.0, 1, 40, 0.5)
xrot_sl  = _sl("X rotation", 0, -180, 180, 1)
yrot_sl  = _sl("Y rotation", 0, -180, 180, 1)
zrot_sl  = _sl("Z rotation", 90, -180, 180, 1)

# ── World ──
bg_color = widgets.ColorPicker(
    value="#ffffff", description="Background:",
    style={"description_width": DESC_W}, layout=SL,
)
fog_front  = _sl("Fog front", 1.0, 0, 2, 0.05)
fog_back   = _sl("Fog back", 1.0, 0, 2, 0.05)
shadows_cb = widgets.Checkbox(value=True, description="Shadows", indent=False,
                               style={"description_width": DESC_W}, layout=SL)
shadow_str = _sl("Strength", 0.0023, 0, 0.01, 0.0001, readout_format=".4f")
shadow_ang = _sl("Angle", 2.0, 0, 10, 0.5)
shadow_dark = _sl("Max dark", 0.2, 0, 1, 0.05)
width_sl  = widgets.IntSlider(value=-30, min=-60, max=800, step=10, description="Width:",
                               style={"description_width": DESC_W, "handle_color": "#7b8cff"},
                               layout=SL, continuous_update=False)
height_sl = widgets.IntSlider(value=-30, min=-60, max=800, step=10, description="Height:",
                               style={"description_width": DESC_W, "handle_color": "#7b8cff"},
                               layout=SL, continuous_update=False)

# ── Outlines ──
outlines_cb    = widgets.Checkbox(value=True, description="Outlines", indent=False,
                                   style={"description_width": DESC_W}, layout=SL)
outline_kernel = widgets.Dropdown(options=[1, 2, 3, 4], value=4, description="Kernel:",
                                   style={"description_width": DESC_W}, layout=SL)
contour_lo  = _sl("Contour lo", 1.0, 0, 10, 0.5)
contour_hi  = _sl("Contour hi", 10.0, 0, 30, 0.5)
z_diff_min  = _sl("Z diff min", 1.0, 0, 20, 0.5)
z_diff_max  = _sl("Z diff max", 50.0, 0, 100, 1)
sub_lo      = _sl("Subunit lo", 3.0, 0, 20, 0.5)
sub_hi      = _sl("Subunit hi", 10.0, 0, 30, 0.5)
res_lo      = _sl("Residue lo", 3.0, 0, 20, 0.5)
res_hi      = _sl("Residue hi", 8.0, 0, 30, 0.5)
res_diff    = _sl("Res diff", 6000.0, 0, 20000, 100)

# ── Actions ──
render_btn = widgets.Button(description="Render", layout=widgets.Layout(width="80px", height="30px"))
render_btn.add_class("ill-btn-render")
save_btn = widgets.Button(description="Save PNG", layout=widgets.Layout(width="80px", height="30px"))
save_btn.add_class("ill-btn-save")
auto_cb = widgets.Checkbox(value=True, description="Auto", indent=False,
                            style={"description_width": "auto"},
                            layout=widgets.Layout(width="70px"))

# ── Status / viewport ──
status_lbl = widgets.HTML(value="<span style='color:#888899'>Enter a PDB ID and click Fetch.</span>")
output_area = widgets.Output()

# ━━━━━━━━━━━━━━━━━━━ Layout ━━━━━━━━━━━━━━━━━━━

fetch_row = widgets.HBox([pdb_input, fetch_btn],
                          layout=widgets.Layout(align_items="center", gap="6px"))

action_row = widgets.HBox([render_btn, save_btn, auto_cb],
                           layout=widgets.Layout(gap="6px", align_items="center"))

sidebar = widgets.VBox([
    _section("\u2b22", "Molecule"),
    fetch_row, preset_dd,
    _section("\u25e4", "Transform"),
    scale_sl, xrot_sl, yrot_sl, zrot_sl,
    _section("\u25cb", "World"),
    bg_color, fog_front, fog_back,
    shadows_cb, shadow_str, shadow_ang, shadow_dark,
    width_sl, height_sl,
    _section("\u2581", "Outlines"),
    outlines_cb, outline_kernel,
    contour_lo, contour_hi, z_diff_min, z_diff_max,
    sub_lo, sub_hi, res_lo, res_hi, res_diff,
    GAP_MD,
    action_row,
], layout=widgets.Layout(
    width="340px", min_width="340px", max_height="640px",
    overflow_y="auto", padding="16px 14px",
))
sidebar.add_class("ill-sidebar")

viewport = widgets.VBox([output_area],
                         layout=widgets.Layout(flex="1", min_height="480px",
                                               align_items="center", justify_content="center",
                                               padding="16px"))
viewport.add_class("ill-viewport")

status_bar = widgets.HBox([status_lbl],
                           layout=widgets.Layout(padding="8px 16px"))
status_bar.add_class("ill-status")

main_row = widgets.HBox([sidebar, viewport],
                         layout=widgets.Layout(width="100%"))

root = widgets.VBox([main_row, status_bar],
                     layout=widgets.Layout(width="100%", border="1px solid #2a2a3e",
                                           border_radius="12px", overflow="hidden"))
root.add_class("ill-root")

# ━━━━━━━━━━━━━━━━━━━ Logic ━━━━━━━━━━━━━━━━━━━

def _sync_from_preset(name):
    global _syncing
    if _pdb_path is None:
        return
    _syncing = True
    try:
        p = render_params_from_preset(name, _pdb_path)
        scale_sl.value = p.transform.scale
        rots = {a: d for a, d in p.transform.rotations}
        xrot_sl.value = rots.get("x", 0)
        yrot_sl.value = rots.get("y", 0)
        zrot_sl.value = rots.get("z", 0)
        bg_color.value = _rgb_to_hex(p.world.background)
        fog_front.value = p.world.fog_front
        fog_back.value = p.world.fog_back
        shadows_cb.value = p.world.shadows
        shadow_str.value = p.world.shadow_strength
        shadow_ang.value = p.world.shadow_angle
        shadow_dark.value = p.world.shadow_max_dark
        width_sl.value = p.world.width
        height_sl.value = p.world.height
        outlines_cb.value = p.outlines.enabled
        outline_kernel.value = p.outlines.kernel
        contour_lo.value = p.outlines.contour_low
        contour_hi.value = p.outlines.contour_high
        z_diff_min.value = p.outlines.z_diff_min
        z_diff_max.value = p.outlines.z_diff_max
        sub_lo.value = p.outlines.subunit_low
        sub_hi.value = p.outlines.subunit_high
        res_lo.value = p.outlines.residue_low
        res_hi.value = p.outlines.residue_high
        res_diff.value = p.outlines.residue_diff
    finally:
        _syncing = False

def _set_status(text, color="#888899"):
    status_lbl.value = f"<span style='color:{color}'>{text}</span>"

def on_fetch(_):
    global _pdb_path
    pdb_id = pdb_input.value.strip()
    if not pdb_id:
        _set_status("Enter a PDB ID.")
        return
    _set_status(f"Fetching {pdb_id}\u2026")
    try:
        _pdb_path = str(fetch_pdb(pdb_id))
        _sync_from_preset(preset_dd.value)
        _set_status(f"\u2713 {pdb_id} loaded", "#7b8cff")
        if _auto_render:
            _do_render()
    except Exception as e:
        _set_status(f"Fetch failed: {e}", "#ef5350")

def _do_render():
    global _last_result
    if _pdb_path is None:
        _set_status("Fetch a PDB first.")
        return
    _set_status("Rendering\u2026")
    try:
        p = render_params_from_preset(preset_dd.value, _pdb_path)
        bg = _hex_to_rgb(bg_color.value)
        params = RenderParams(
            pdb_path=p.pdb_path,
            rules=p.rules,
            transform=Transform(
                scale=scale_sl.value,
                rotations=[("z", zrot_sl.value), ("y", yrot_sl.value), ("x", xrot_sl.value)],
                autocenter="auto",
            ),
            world=WorldParams(
                background=bg, fog_color=bg,
                fog_front=fog_front.value, fog_back=fog_back.value,
                shadows=shadows_cb.value,
                shadow_strength=shadow_str.value,
                shadow_angle=shadow_ang.value,
                shadow_min_z=p.world.shadow_min_z,
                shadow_max_dark=shadow_dark.value,
                width=width_sl.value, height=height_sl.value,
            ),
            outlines=OutlineParams(
                enabled=outlines_cb.value,
                contour_low=contour_lo.value, contour_high=contour_hi.value,
                kernel=outline_kernel.value,
                z_diff_min=z_diff_min.value, z_diff_max=z_diff_max.value,
                subunit_low=sub_lo.value, subunit_high=sub_hi.value,
                residue_low=res_lo.value, residue_high=res_hi.value,
                residue_diff=res_diff.value,
            ),
        )
        _last_result = render(params)
        img = Image.fromarray(_last_result.rgb)
        with output_area:
            clear_output(wait=True)
            display(img)
        _set_status(f"\u2713 {_last_result.width}\u00d7{_last_result.height}", "#66bb6a")
    except Exception as e:
        _set_status(f"Error: {e}", "#ef5350")

def on_render(_):
    _do_render()

def on_save(_):
    if _last_result is None:
        _set_status("Nothing to save \u2014 render first.")
        return
    write_png("illustrate_output.png", _last_result.rgb)
    _set_status("\u2713 Saved illustrate_output.png", "#66bb6a")

def on_auto_toggle(change):
    global _auto_render
    _auto_render = change["new"]

def on_preset_change(change):
    _sync_from_preset(change["new"])
    if _auto_render and _pdb_path:
        _do_render()

def _on_param_change(_change=None):
    if _syncing:
        return
    if _auto_render and _pdb_path:
        _do_render()

# ── Wire up ──
fetch_btn.on_click(on_fetch)
render_btn.on_click(on_render)
save_btn.on_click(on_save)
auto_cb.observe(on_auto_toggle, names="value")
preset_dd.observe(on_preset_change, names="value")

for w in [scale_sl, xrot_sl, yrot_sl, zrot_sl,
          bg_color, fog_front, fog_back,
          shadows_cb, shadow_str, shadow_ang, shadow_dark,
          width_sl, height_sl,
          outlines_cb, outline_kernel,
          contour_lo, contour_hi, z_diff_min, z_diff_max,
          sub_lo, sub_hi, res_lo, res_hi, res_diff]:
    w.observe(_on_param_change, names="value")

display(root)